# Pair-K comparison

This notebook prepares the `k=1`, `k=2`, and `k=3` comparison for pairwise ranking.

Steps:
1. generate deterministic pair sets for `k=1,2,3`
2. filter the existing `pairwise_labels.jsonl` by membership in those generated pair sets
3. regenerate manifests for each filtered pairwise-label file with the repo manifest builder
4. sanity-check the outputs

So the source of truth here is the existing pairwise labels, and each `k` gets its own freshly regenerated manifests.


## Step 1. Generate candidate pair sets

Generate the `k=1`, `k=2`, and `k=3` pair sets directly through the repo pair-generation function. This writes local pair files under `data/pairs_k_comparison/pairs/`.


In [2]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from aitraf.data_ops.create_pairwise_manifests import (
    PairwiseManifestBuildConfig,
    PairwiseTaskConfig,
    create_pairwise_manifests,
)
from aitraf.datasets.score_prediction_rank import ScorePredictionRankDataset
from aitraf.label_ops.create_pairs import PairGenerationConfig, create_pairs

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
EXPERIMENT_DIR = DATA_DIR / "pairs_k_comparison"
PAIRS_BASE_DIR = EXPERIMENT_DIR / "pairs"
FILTERED_PAIRWISE_LABELS_DIR = EXPERIMENT_DIR / "pairwise_labels"
OUTPUT_MANIFESTS_BASE_DIR = EXPERIMENT_DIR / "manifests"
PAIRWISE_LABELS_PATH = DATA_DIR / "pairwise_labels.jsonl"
LABELS_PATH = DATA_DIR / "labels.jsonl"
K_VALUES = (1, 2, 3)
PAIR_SEED = 42
SPLIT_SEED = 42
VAL_RATIO = 0.1
TEST_RATIO = 0.1

for k in K_VALUES:
    create_pairs(
        PairGenerationConfig(
            labels_path=LABELS_PATH,
            output_dir=PAIRS_BASE_DIR / f"k{k}",
            k_per_video=k,
            seed=PAIR_SEED,
            force=True,
        )
    )


2026-03-31 12:22:57.822 | INFO     | aitraf.label_ops.create_pairs:create_pairs:102 - Trick 'ao-soul' -> 156 new pairs
2026-03-31 12:22:57.831 | INFO     | aitraf.label_ops.create_pairs:create_pairs:102 - Trick 'bs-royale' -> 183 new pairs
2026-03-31 12:22:57.837 | INFO     | aitraf.label_ops.create_pairs:create_pairs:102 - Trick 'fs-royale' -> 157 new pairs
2026-03-31 12:22:57.845 | INFO     | aitraf.label_ops.create_pairs:create_pairs:102 - Trick 'fs-savanah' -> 168 new pairs
2026-03-31 12:22:57.852 | INFO     | aitraf.label_ops.create_pairs:create_pairs:102 - Trick 'mizou' -> 146 new pairs
2026-03-31 12:22:57.861 | INFO     | aitraf.label_ops.create_pairs:create_pairs:102 - Trick 'soul' -> 164 new pairs
2026-03-31 12:22:57.868 | INFO     | aitraf.label_ops.create_pairs:create_pairs:102 - Trick 'top-soul' -> 142 new pairs
2026-03-31 12:22:57.869 | INFO     | aitraf.label_ops.create_pairs:create_pairs:104 - Created 1116 new pair files in /workspace/data/pairs_k_comparison/pairs/k1 (0 

## Step 2. Load the existing pairwise labels and generated pair sets

The generated pair dirs act as filters. The labeled source is `data/pairwise_labels.jsonl`.


In [3]:
pairwise_labels_df = pd.read_json(PAIRWISE_LABELS_PATH, lines=True)


def canonical_pair(left: str, right: str) -> tuple[str, str]:
    return (left, right) if left <= right else (right, left)


def load_generated_pair_keys(pairs_dir: Path) -> set[tuple[str, str]]:
    pair_keys: set[tuple[str, str]] = set()
    for path in sorted(pairs_dir.glob("*.json")):
        payload = json.loads(path.read_text())
        data = payload.get("data") or {}
        pair_keys.add(canonical_pair(data["left"], data["right"]))
    return pair_keys


pair_keys_by_k = {
    k: load_generated_pair_keys(PAIRS_BASE_DIR / f"k{k}")
    for k in K_VALUES
}

print(f"pairwise labels: {len(pairwise_labels_df)} rows")
for k, pair_keys in pair_keys_by_k.items():
    print(f"  k={k}: {len(pair_keys)} generated pairs")


pairwise labels: 315 rows
  k=1: 1116 generated pairs
  k=2: 1663 generated pairs
  k=3: 2210 generated pairs


## Step 3. Filter and save `pairwise_labels` for each `k`

For each `k`, keep only the rows whose `(left, right)` pair exists in that generated pair set.


In [4]:
FILTERED_PAIRWISE_LABELS_DIR.mkdir(parents=True, exist_ok=True)

filtered_label_rows = []

for k in K_VALUES:
    filtered_df = pairwise_labels_df.loc[
        pairwise_labels_df.apply(
            lambda row: canonical_pair(row["left"], row["right"]) in pair_keys_by_k[k],
            axis=1,
        )
    ].reset_index(drop=True)

    output_path = FILTERED_PAIRWISE_LABELS_DIR / f"k{k}.jsonl"
    filtered_df.to_json(output_path, orient="records", lines=True, force_ascii=False)

    filtered_label_rows.append(
        {
            "k": k,
            "generated_pairs": len(pair_keys_by_k[k]),
            "filtered_pairwise_labels": len(filtered_df),
            "output_path": str(output_path),
        }
    )

display(pd.DataFrame(filtered_label_rows).sort_values("k").reset_index(drop=True))


,k,generated_pairs,filtered_pairwise_labels,output_path
0,1,1116,84,/workspace/data/pairs_k_comparison/pairwise_la...
1,2,1663,95,/workspace/data/pairs_k_comparison/pairwise_la...
2,3,2210,313,/workspace/data/pairs_k_comparison/pairwise_la...


## Step 4. Regenerate manifests for each `k`

This uses the repo pairwise manifest builder, so each `k` gets a fresh `train/val/test` split from its filtered `pairwise_labels` file.


In [5]:
for k in K_VALUES:
    manifests_dir = OUTPUT_MANIFESTS_BASE_DIR / f"k{k}"
    pairwise_labels_path = FILTERED_PAIRWISE_LABELS_DIR / f"k{k}.jsonl"

    create_pairwise_manifests(
        PairwiseManifestBuildConfig(
            output_dir=OUTPUT_MANIFESTS_BASE_DIR,
            val_ratio=VAL_RATIO,
            test_ratio=TEST_RATIO,
            split_seed=SPLIT_SEED,
            force=True,
            tasks=(
                PairwiseTaskConfig(
                    name=f"k{k}",
                    pairwise_labels_path=pairwise_labels_path,
                    labels_path=LABELS_PATH,
                    pairwise_labels_required_cols=("left", "right", "pref"),
                    labels_required_cols=("video", "trick"),
                    manifests_dir=manifests_dir,
                    vocab_path=manifests_dir / "vocab.json",
                    stratify_col=None,
                ),
            ),
        )
    )


2026-03-31 12:22:58.246 | WARNING  | aitraf.data_ops.create_pairwise_manifests:_build_pairwise_manifest_df:203 - Dropping 1 pairwise label rows without a selected preference
2026-03-31 12:22:58.255 | INFO     | aitraf.data_ops.create_pairwise_manifests:_write_vocab:334 - Wrote categorical vocab to /workspace/data/pairs_k_comparison/manifests/k1/vocab.json
2026-03-31 12:22:58.260 | INFO     | aitraf.data_ops.create_pairwise_manifests:_build_task_manifests:161 - Task 'k1' wrote /workspace/data/pairs_k_comparison/manifests/k1/train.jsonl (65 rows)
2026-03-31 12:22:58.261 | INFO     | aitraf.data_ops.create_pairwise_manifests:_build_task_manifests:161 - Task 'k1' wrote /workspace/data/pairs_k_comparison/manifests/k1/val.jsonl (9 rows)
2026-03-31 12:22:58.262 | INFO     | aitraf.data_ops.create_pairwise_manifests:_build_task_manifests:161 - Task 'k1' wrote /workspace/data/pairs_k_comparison/manifests/k1/test.jsonl (9 rows)
2026-03-31 12:22:58.282 | WARNING  | aitraf.data_ops.create_pairwise

## Step 5. Quick sanity check

This just shows the size of the regenerated `train/val/test` manifests for each `k`.


In [6]:
rows = []

for k in K_VALUES:
    manifest_dir = OUTPUT_MANIFESTS_BASE_DIR / f"k{k}"
    for split in ["train", "val", "test"]:
        df = pd.DataFrame(
            ScorePredictionRankDataset(manifests_dir=manifest_dir, split=split).manifest_rows()
        )
        rows.append({"k": k, "split": split, "rows": len(df)})

display(pd.DataFrame(rows).sort_values(["k", "split"]).reset_index(drop=True))


,k,split,rows
0,1,test,9
1,1,train,65
2,1,val,9
3,2,test,10
4,2,train,74
5,2,val,10
6,3,test,31
7,3,train,247
8,3,val,32
